In [1]:

# Importer les bibliothèques nécessaires au Job Search Agent V2.
# Charger les identifiants JSearch et configurer les URL des APIs.

import os
import re
import requests
import unicodedata

from pathlib import Path
from datetime import datetime, timezone
from html import unescape
from html.parser import HTMLParser
from collections import defaultdict
from typing import Optional, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel, Field, HttpUrl, ValidationError
from langgraph.graph import StateGraph, START, END

# Charger le fichier .env
env_path = Path.cwd().parent / ".env"
load_dotenv(env_path, override=True)

# Identifiants JSearch
RAPIDAPI_KEY = os.getenv("RAPIDAPI_KEY")
RAPIDAPI_HOST = os.getenv("RAPIDAPI_HOST", "jsearch.p.rapidapi.com")

# URLs des APIs
JSEARCH_API_URL = "https://jsearch.p.rapidapi.com/search-v2"
REMOTIVE_API_URL = "https://remotive.com/api/remote-jobs"
ARBEITNOW_API_URL = "https://www.arbeitnow.com/api/job-board-api"
GREENHOUSE_API_URL = "https://boards-api.greenhouse.io/v1/boards"

# Vérification
print("=== CONFIGURATION JOB SEARCH AGENT V2 ===")
print("Fichier .env trouvé :", env_path.is_file())
print("JSearch configuré :", bool(RAPIDAPI_KEY))
print("Host JSearch :", RAPIDAPI_HOST)
print("Remotive : OK")
print("Arbeitnow : OK")
print("Greenhouse : OK")


=== CONFIGURATION JOB SEARCH AGENT V2 ===
Fichier .env trouvé : True
JSearch configuré : True
Host JSearch : jsearch.p.rapidapi.com
Remotive : OK
Arbeitnow : OK
Greenhouse : OK


In [2]:

# Définir les modèles Pydantic pour les offres et les critères de recherche.
# Garantir une structure commune compatible avec le Matching Agent.

class JobOffer(BaseModel):
    source: str
    source_id: str

    title: str
    company: str
    location: Optional[str] = None
    description: Optional[str] = None

    tags: list[str] = Field(default_factory=list)
    job_type: Optional[str] = None
    category: Optional[str] = None
    salary: Optional[str] = None

    publication_date: Optional[str] = None
    url: HttpUrl


class JobSearchFilters(BaseModel):
    keywords: list[str] = Field(default_factory=list)
    country: str = "tn"
    location: Optional[str] = None
    remote_only: bool = False
    employment_type: Optional[str] = None
    max_results_per_source: int = Field(default=20, ge=1, le=100)


# Initialiser les filtres de test
search_filters = JobSearchFilters(
    keywords=["Python Developer", "AI Engineer"],
    country="tn",
    location=None,
    remote_only=False,
    employment_type=None,
    max_results_per_source=20
)

# Vérification
print("=== MODÈLES PYDANTIC ===")
print("JobOffer : OK")
print("JobSearchFilters : OK")
print("\nFiltres :", search_filters.model_dump())


=== MODÈLES PYDANTIC ===
JobOffer : OK
JobSearchFilters : OK

Filtres : {'keywords': ['Python Developer', 'AI Engineer'], 'country': 'tn', 'location': None, 'remote_only': False, 'employment_type': None, 'max_results_per_source': 20}


In [3]:

# Convertir les offres brutes Remotive en objets JobOffer.
# Ignorer les offres invalides sans interrompre le traitement.

def normalize_remotive_jobs(raw_jobs: list[dict]) -> list[JobOffer]:

    normalized_jobs = []

    for job in raw_jobs:
        try:
            offer = JobOffer(
                source="remotive",
                source_id=str(job["id"]),
                title=job["title"],
                company=job["company_name"],
                location=job.get("candidate_required_location"),
                description=job.get("description"),
                tags=job.get("tags") or [],
                job_type=job.get("job_type"),
                category=job.get("category"),
                salary=job.get("salary"),
                publication_date=job.get("publication_date"),
                url=job["url"]
            )

            normalized_jobs.append(offer)

        except (ValidationError, KeyError, TypeError) as error:
            print(f"Offre Remotive ignorée : {error}")

    return normalized_jobs


# Test local sans appel API
test_remotive_jobs = [
    {
        "id": 12345,
        "title": "Python Developer",
        "company_name": "Tech Company",
        "candidate_required_location": "Worldwide",
        "description": "<p>Python, FastAPI, PostgreSQL</p>",
        "tags": ["Python", "FastAPI"],
        "job_type": "full_time",
        "category": "Software Development",
        "salary": None,
        "publication_date": "2026-10-01",
        "url": "https://example.com/jobs/12345"
    }
]

remotive_offers = normalize_remotive_jobs(test_remotive_jobs)

print("=== TEST NORMALISATION REMOTIVE ===")
print("Offres normalisées :", len(remotive_offers))
print(remotive_offers[0].model_dump_json(indent=2))


=== TEST NORMALISATION REMOTIVE ===
Offres normalisées : 1
{
  "source": "remotive",
  "source_id": "12345",
  "title": "Python Developer",
  "company": "Tech Company",
  "location": "Worldwide",
  "description": "<p>Python, FastAPI, PostgreSQL</p>",
  "tags": [
    "Python",
    "FastAPI"
  ],
  "job_type": "full_time",
  "category": "Software Development",
  "salary": null,
  "publication_date": "2026-10-01",
  "url": "https://example.com/jobs/12345"
}


In [4]:

# Convertir les offres Arbeitnow en objets JobOffer.
# Harmoniser les types de contrat et les dates de publication.

def normalize_arbeitnow_jobs(raw_jobs: list[dict]) -> list[JobOffer]:

    normalized_jobs = []

    for job in raw_jobs:
        try:
            job_types = job.get("job_types") or []
            created_at = job.get("created_at")

            publication_date = (
                datetime.fromtimestamp(
                    created_at,
                    tz=timezone.utc
                ).isoformat()
                if created_at is not None
                else None
            )

            offer = JobOffer(
                source="arbeitnow",
                source_id=str(job["slug"]),
                title=job["title"],
                company=job["company_name"],
                location=job.get("location") or None,
                description=job.get("description"),
                tags=job.get("tags") or [],
                job_type=", ".join(job_types) if job_types else None,
                category=None,
                salary=None,
                publication_date=publication_date,
                url=job["url"]
            )

            normalized_jobs.append(offer)

        except (
            ValidationError,
            KeyError,
            TypeError,
            ValueError,
            OverflowError
        ) as error:
            print(f"Offre Arbeitnow ignorée : {error}")

    return normalized_jobs


# Test local sans appel API
test_arbeitnow_jobs = [
    {
        "slug": "backend-python-developer",
        "title": "Backend Python Developer",
        "company_name": "Example Tech",
        "location": "Berlin",
        "description": "<p>Python et Django</p>",
        "tags": ["Python", "Django"],
        "job_types": ["full_time"],
        "created_at": 1790812800,
        "url": "https://example.com/jobs/backend-python"
    }
]

arbeitnow_offers = normalize_arbeitnow_jobs(test_arbeitnow_jobs)

print("=== TEST NORMALISATION ARBEITNOW ===")
print("Offres normalisées :", len(arbeitnow_offers))
print(arbeitnow_offers[0].model_dump_json(indent=2))


=== TEST NORMALISATION ARBEITNOW ===
Offres normalisées : 1
{
  "source": "arbeitnow",
  "source_id": "backend-python-developer",
  "title": "Backend Python Developer",
  "company": "Example Tech",
  "location": "Berlin",
  "description": "<p>Python et Django</p>",
  "tags": [
    "Python",
    "Django"
  ],
  "job_type": "full_time",
  "category": null,
  "salary": null,
  "publication_date": "2026-10-01T00:00:00+00:00",
  "url": "https://example.com/jobs/backend-python"
}


In [31]:

# Récupérer les offres brutes depuis l'API Arbeitnow.
# Filtrer les résultats selon les métiers et préférences du candidat.

def fetch_arbeitnow_jobs(filters: JobSearchFilters) -> list[dict]:

    response = requests.get(
        ARBEITNOW_API_URL,
        timeout=20
    )
    response.raise_for_status()

    raw_jobs = response.json().get("data", [])
    filtered_jobs = []

    for job in raw_jobs:
        title = (job.get("title") or "").casefold()
        description = (job.get("description") or "").casefold()
        location = (job.get("location") or "").casefold()

        if filters.keywords and not any(
            keyword.casefold() in title + " " + description
            for keyword in filters.keywords
        ):
            continue

        if filters.location and filters.location.casefold() not in location:
            continue

        if filters.remote_only and not job.get("remote", False):
            continue

        if filters.employment_type:
            job_types = job.get("job_types") or []
            if not any(
                filters.employment_type.casefold() == str(t).casefold()
                for t in job_types
            ):
                continue

        filtered_jobs.append(job)

        if len(filtered_jobs) >= filters.max_results_per_source:
            break

    return filtered_jobs


print("✅ fetch_arbeitnow_jobs définie !")
print("Fonction disponible :", "fetch_arbeitnow_jobs" in globals())


✅ fetch_arbeitnow_jobs définie !
Fonction disponible : True


In [5]:

# Convertir les offres Greenhouse vers le modèle commun JobOffer.
# Extraire les informations essentielles et nettoyer les descriptions HTML.

def normalize_greenhouse_jobs(
    raw_jobs: list[dict],
    company_name: str
) -> list[JobOffer]:

    normalized_jobs = []

    for job in raw_jobs:
        try:
            job_url = job.get("absolute_url")

            if not job_url:
                continue

            description_html = job.get("content") or ""
            description_text = re.sub(
                r"<[^>]+>",
                " ",
                description_html
            )
            description_text = unescape(description_text)
            description_text = " ".join(description_text.split())

            location = (job.get("location") or {}).get("name")
            departments = job.get("departments") or []

            tags = [
                department.get("name")
                for department in departments
                if department.get("name")
            ]

            offer = JobOffer(
                source="greenhouse",
                source_id=str(job["id"]),
                title=job.get("title") or "Poste non précisé",
                company=company_name,
                location=location,
                description=description_text,
                tags=tags,
                job_type=None,
                category=None,
                salary=None,
                publication_date=job.get("updated_at"),
                url=job_url
            )

            normalized_jobs.append(offer)

        except (ValidationError, KeyError, TypeError) as error:
            print(f"Offre Greenhouse ignorée : {error}")

    return normalized_jobs


# Test local sans appel API
test_greenhouse_jobs = [
    {
        "id": 98765,
        "title": "AI Engineer",
        "location": {"name": "Remote"},
        "content": "<p>Python, Machine Learning et LangGraph</p>",
        "departments": [{"name": "Engineering"}],
        "updated_at": "2026-10-01T10:00:00Z",
        "absolute_url": "https://example.com/jobs/ai-engineer"
    }
]

greenhouse_offers = normalize_greenhouse_jobs(
    test_greenhouse_jobs,
    company_name="Example AI"
)

print("=== TEST NORMALISATION GREENHOUSE ===")
print("Offres normalisées :", len(greenhouse_offers))

if greenhouse_offers:
    print(greenhouse_offers[0].model_dump_json(indent=2))


=== TEST NORMALISATION GREENHOUSE ===
Offres normalisées : 1
{
  "source": "greenhouse",
  "source_id": "98765",
  "title": "AI Engineer",
  "company": "Example AI",
  "location": "Remote",
  "description": "Python, Machine Learning et LangGraph",
  "tags": [
    "Engineering"
  ],
  "job_type": null,
  "category": null,
  "salary": null,
  "publication_date": "2026-10-01T10:00:00Z",
  "url": "https://example.com/jobs/ai-engineer"
}


In [6]:

# Convertir les offres JSearch V2 vers le modèle commun JobOffer.
# Conserver les informations essentielles et les liens de candidature.

def normalize_jsearch_jobs(
    raw_jobs: list[dict]
) -> list[JobOffer]:

    normalized_jobs = []

    for job in raw_jobs:
        try:
            apply_url = job.get("job_apply_link")

            if not apply_url:
                continue

            offer = JobOffer(
                source="jsearch",
                source_id=str(
                    job.get("job_id") or apply_url
                ),
                title=job.get("job_title") or "Titre inconnu",
                company=job.get("employer_name") or "Entreprise inconnue",
                location=(
                    job.get("job_location")
                    or job.get("job_city")
                    or job.get("job_country")
                ),
                description=job.get("job_description"),
                tags=(
                    ["remote"]
                    if job.get("job_is_remote")
                    else []
                ),
                job_type=job.get("job_employment_type"),
                category=None,
                salary=None,
                publication_date=job.get("job_posted_at_datetime_utc"),
                url=apply_url
            )

            normalized_jobs.append(offer)

        except (ValidationError, TypeError, ValueError) as error:
            print(f"Offre JSearch ignorée : {error}")

    return normalized_jobs


# Test local sans appel API
test_jsearch_jobs = [
    {
        "job_id": "jsearch-123",
        "job_title": "Python AI Engineer",
        "employer_name": "Example AI",
        "job_location": "Tunis, Tunisia",
        "job_description": "Python, FastAPI, LangGraph",
        "job_is_remote": False,
        "job_employment_type": "FULLTIME",
        "job_posted_at_datetime_utc": "2026-10-01T10:00:00Z",
        "job_apply_link": "https://example.com/jobs/python-ai"
    }
]

jsearch_offers = normalize_jsearch_jobs(test_jsearch_jobs)

print("=== TEST NORMALISATION JSEARCH ===")
print("Offres normalisées :", len(jsearch_offers))

if jsearch_offers:
    print(jsearch_offers[0].model_dump_json(indent=2))


=== TEST NORMALISATION JSEARCH ===
Offres normalisées : 1
{
  "source": "jsearch",
  "source_id": "jsearch-123",
  "title": "Python AI Engineer",
  "company": "Example AI",
  "location": "Tunis, Tunisia",
  "description": "Python, FastAPI, LangGraph",
  "tags": [],
  "job_type": "FULLTIME",
  "category": null,
  "salary": null,
  "publication_date": "2026-10-01T10:00:00Z",
  "url": "https://example.com/jobs/python-ai"
}


In [7]:

# Nettoyer les localisations JSearch en supprimant les noms des plateformes.
# Conserver uniquement la ville, la région ou le pays de l'offre.

def clean_jsearch_location(location: Optional[str]) -> Optional[str]:

    if not location:
        return None

    cleaned_location = re.split(
        r"\s*[•·]\s*(?:عبر|via)\s+",
        location,
        maxsplit=1,
        flags=re.IGNORECASE
    )[0]

    return " ".join(cleaned_location.split()) or None


# Test local sans appel API
test_locations = [
    "تونس • عبر Trabajo.org",
    "Tunis • via LinkedIn",
    "Paris, France",
    "Remote",
    None
]

print("=== TEST NETTOYAGE LOCALISATIONS ===")

for location in test_locations:
    print(f"{location} → {clean_jsearch_location(location)}")


=== TEST NETTOYAGE LOCALISATIONS ===
تونس • عبر Trabajo.org → تونس
Tunis • via LinkedIn → Tunis
Paris, France → Paris, France
Remote → Remote
None → None


In [8]:

# Extraire le texte des descriptions HTML des offres d'emploi.
# Supprimer les balises inutiles et préparer les textes pour le Matching Agent.

class HTMLTextExtractor(HTMLParser):

    def __init__(self):
        super().__init__()
        self.parts = []
        self.skip_depth = 0

    def handle_starttag(self, tag, attrs):
        if tag in ("script", "style"):
            self.skip_depth += 1

        elif self.skip_depth == 0 and tag in (
            "p", "br", "li", "div", "h1", "h2", "h3"
        ):
            self.parts.append("\n")

    def handle_endtag(self, tag):
        if tag in ("script", "style"):
            self.skip_depth = max(0, self.skip_depth - 1)

        elif self.skip_depth == 0 and tag in (
            "p", "li", "div", "h1", "h2", "h3"
        ):
            self.parts.append("\n")

    def handle_data(self, data):
        if self.skip_depth == 0:
            self.parts.append(data)


def clean_html_description(
    description: Optional[str]
) -> str:

    if not description:
        return ""

    parser = HTMLTextExtractor()
    parser.feed(description)

    text = unescape("".join(parser.parts))

    lines = [
        " ".join(line.split())
        for line in text.splitlines()
    ]

    return "\n".join(
        line for line in lines if line
    )


# Test local sans appel API
test_description = """
<h2>Python Developer</h2>
<p>Nous recherchons un développeur Python.</p>
<ul>
    <li>Python</li>
    <li>FastAPI</li>
    <li>PostgreSQL</li>
</ul>
<script>alert("test")</script>
"""

cleaned_description = clean_html_description(test_description)

print("=== DESCRIPTION NETTOYÉE ===")
print(cleaned_description)


=== DESCRIPTION NETTOYÉE ===
Python Developer
Nous recherchons un développeur Python.
Python
FastAPI
PostgreSQL


In [9]:

# Nettoyer les descriptions de toutes les offres normalisées.
# Préserver les objets originaux avant leur transmission au Matching Agent.

def prepare_jobs_for_matching(
    offers: list[JobOffer]
) -> list[JobOffer]:

    cleaned_offers = []

    for offer in offers:

        cleaned_offer = offer.model_copy(
            update={
                "description": clean_html_description(
                    offer.description
                )
            }
        )

        cleaned_offers.append(cleaned_offer)

    return cleaned_offers


# Test local sans appel API
test_offers = remotive_offers + arbeitnow_offers + greenhouse_offers + jsearch_offers

prepared_offers = prepare_jobs_for_matching(test_offers)

print("=== TEST PRÉPARATION DES OFFRES ===")
print("Offres initiales :", len(test_offers))
print("Offres préparées :", len(prepared_offers))

for offer in prepared_offers:
    print("\n---------------------------")
    print("Source :", offer.source)
    print("Titre :", offer.title)
    print("Description :", offer.description)


=== TEST PRÉPARATION DES OFFRES ===
Offres initiales : 4
Offres préparées : 4

---------------------------
Source : remotive
Titre : Python Developer
Description : Python, FastAPI, PostgreSQL

---------------------------
Source : arbeitnow
Titre : Backend Python Developer
Description : Python et Django

---------------------------
Source : greenhouse
Titre : AI Engineer
Description : Python, Machine Learning et LangGraph

---------------------------
Source : jsearch
Titre : Python AI Engineer
Description : Python, FastAPI, LangGraph


In [41]:

# Rechercher séparément chaque métier sur JSearch pour améliorer les résultats.
# Normaliser, nettoyer les localisations et supprimer les doublons.

def fetch_jsearch_jobs(
    filters: JobSearchFilters
) -> list[JobOffer]:

    if not RAPIDAPI_KEY:
        raise ValueError("Clé RAPIDAPI_KEY manquante")

    location = filters.location or (
        "Tunisia" if filters.country.lower() == "tn"
        else filters.country
    )

    headers = {
        "x-rapidapi-key": RAPIDAPI_KEY,
        "x-rapidapi-host": RAPIDAPI_HOST
    }

    all_offers = []
    seen_ids = set()

    for role in filters.keywords:

        print(f"Recherche JSearch : {role}")

        params = {
            "query": f"{role} in {location}",
            "country": filters.country,
            "num_pages": 1
        }

        if filters.remote_only:
            params["work_from_home"] = "true"

        response = requests.get(
            JSEARCH_API_URL,
            headers=headers,
            params=params,
            timeout=40
        )

        response.raise_for_status()

        data = response.json()
        payload = data.get("data", {})

        if isinstance(payload, dict):
            raw_jobs = payload.get("jobs", [])
        elif isinstance(payload, list):
            raw_jobs = payload
        else:
            raw_jobs = []

        print(f"  Offres brutes : {len(raw_jobs)}")

        offers = normalize_jsearch_jobs(raw_jobs)

        for offer in offers:

            offer.location = clean_jsearch_location(
                offer.location
            )

            if filters.employment_type:
                if (
                    not offer.job_type
                    or offer.job_type.casefold()
                    != filters.employment_type.casefold()
                ):
                    continue

            unique_key = offer.source_id

            if unique_key in seen_ids:
                continue

            seen_ids.add(unique_key)
            all_offers.append(offer)

            if len(all_offers) >= filters.max_results_per_source:
                break

        if len(all_offers) >= filters.max_results_per_source:
            break

    return all_offers


print("✅ Nouvelle fonction JSearch définie !")


✅ Nouvelle fonction JSearch définie !


In [42]:

# Vérifier que la fonction JSearch corrigée est bien disponible.
# Contrôler les paramètres sans effectuer de requête API.

import inspect

print("=== VÉRIFICATION JSEARCH ===")

print(
    "Fonction disponible :",
    "fetch_jsearch_jobs" in globals()
)

print(
    "Fonction utilisée par le nœud :",
    "fetch_jsearch_jobs" in search_jsearch.__code__.co_names
)

source = inspect.getsource(fetch_jsearch_jobs)

print(
    "Recherche par métier :",
    "for role in filters.keywords" in source
)

print(
    "Nettoyage localisation :",
    "clean_jsearch_location" in source
)

print(
    "Suppression des doublons :",
    "seen_ids" in source
)


=== VÉRIFICATION JSEARCH ===
Fonction disponible : True
Fonction utilisée par le nœud : True
Recherche par métier : True
Nettoyage localisation : True
Suppression des doublons : True


In [11]:

# Rechercher les offres Remotive pour chaque métier ciblé.
# Regrouper les résultats et supprimer les doublons par identifiant.

def fetch_remotive_jobs(
    filters: JobSearchFilters
) -> list[dict]:

    collected_jobs = []
    seen_ids = set()

    for keyword in filters.keywords:

        print(f"Recherche Remotive : {keyword}")

        response = requests.get(
            REMOTIVE_API_URL,
            params={
                "search": keyword,
                "limit": filters.max_results_per_source
            },
            timeout=20
        )

        response.raise_for_status()

        raw_jobs = response.json().get("jobs", [])

        for job in raw_jobs:
            job_id = job.get("id")

            if job_id is None or job_id in seen_ids:
                continue

            if filters.remote_only and not job.get("candidate_required_location"):
                continue

            seen_ids.add(job_id)
            collected_jobs.append(job)

            if len(collected_jobs) >= filters.max_results_per_source:
                break

        if len(collected_jobs) >= filters.max_results_per_source:
            break

    return collected_jobs


print("✅ Fonction fetch_remotive_jobs définie avec succès !")


✅ Fonction fetch_remotive_jobs définie avec succès !


In [12]:

# Rechercher les offres Remotive pour chaque métier ciblé.
# Regrouper les résultats et supprimer les doublons par identifiant.

def fetch_remotive_jobs(
    filters: JobSearchFilters
) -> list[dict]:

    collected_jobs = []
    seen_ids = set()

    for keyword in filters.keywords:

        print(f"Recherche Remotive : {keyword}")

        response = requests.get(
            REMOTIVE_API_URL,
            params={
                "search": keyword,
                "limit": filters.max_results_per_source
            },
            timeout=20
        )

        response.raise_for_status()

        raw_jobs = response.json().get("jobs", [])

        for job in raw_jobs:
            job_id = job.get("id")

            if job_id is None or job_id in seen_ids:
                continue

            if filters.remote_only and not job.get("candidate_required_location"):
                continue

            seen_ids.add(job_id)
            collected_jobs.append(job)

            if len(collected_jobs) >= filters.max_results_per_source:
                break

        if len(collected_jobs) >= filters.max_results_per_source:
            break

    return collected_jobs


print("✅ Fonction fetch_remotive_jobs définie avec succès !")


✅ Fonction fetch_remotive_jobs définie avec succès !


In [13]:

# Filtrer les offres Greenhouse selon les métiers et la localisation recherchés.
# Limiter le nombre de résultats avant leur normalisation.

def filter_greenhouse_jobs(
    jobs: list[dict],
    filters: JobSearchFilters
) -> list[dict]:

    filtered_jobs = []

    keywords = [
        keyword.casefold()
        for keyword in filters.keywords
    ]

    for job in jobs:

        title = job.get("title") or ""

        location = (
            (job.get("location") or {}).get("name") or ""
        )

        # Vérifier le métier recherché
        if keywords and not any(
            keyword in title.casefold()
            for keyword in keywords
        ):
            continue

        # Vérifier la localisation
        if filters.location:
            if filters.location.casefold() not in location.casefold():
                continue

        # Vérifier le télétravail
        if filters.remote_only:
            remote_terms = ("remote", "télétravail", "work from home")

            if not any(
                term in location.casefold()
                for term in remote_terms
            ):
                continue

        filtered_jobs.append(job)

        if len(filtered_jobs) >= filters.max_results_per_source:
            break

    return filtered_jobs


# Test local sans appel API
test_greenhouse_jobs = [
    {
        "id": 101,
        "title": "Python Developer",
        "location": {"name": "Remote"}
    },
    {
        "id": 102,
        "title": "Marketing Manager",
        "location": {"name": "Paris"}
    },
    {
        "id": 103,
        "title": "AI Engineer",
        "location": {"name": "Tunis"}
    }
]

filtered_test = filter_greenhouse_jobs(
    test_greenhouse_jobs,
    search_filters
)

print("=== TEST FILTRE GREENHOUSE ===")

for job in filtered_test:
    print(job["title"], "-", job["location"]["name"])


=== TEST FILTRE GREENHOUSE ===
Python Developer - Remote
AI Engineer - Tunis


In [14]:

# Récupérer les offres publiées sur les pages Greenhouse des entreprises.
# Filtrer puis normaliser les offres pour obtenir des objets JobOffer.

def fetch_greenhouse_jobs(
    filters: JobSearchFilters,
    companies: Optional[dict[str, str]] = None
) -> list[JobOffer]:

    if companies is None:
        companies = {
            "stripe": "Stripe"
        }

    collected_offers = []
    seen_ids = set()

    for board_token, company_name in companies.items():

        url = f"{GREENHOUSE_API_URL}/{board_token}/jobs"

        response = requests.get(
            url,
            params={"content": "true"},
            timeout=30
        )

        response.raise_for_status()

        raw_jobs = response.json().get("jobs", [])

        # Filtrer les offres selon le profil du candidat
        filtered_jobs = filter_greenhouse_jobs(
            raw_jobs,
            filters
        )

        # Transformer les offres en objets JobOffer
        normalized_offers = normalize_greenhouse_jobs(
            filtered_jobs,
            company_name
        )

        for offer in normalized_offers:

            if offer.source_id in seen_ids:
                continue

            seen_ids.add(offer.source_id)
            collected_offers.append(offer)

            if len(collected_offers) >= filters.max_results_per_source:
                break

        if len(collected_offers) >= filters.max_results_per_source:
            break

    return collected_offers


print("✅ Fonction fetch_greenhouse_jobs définie avec succès !")


✅ Fonction fetch_greenhouse_jobs définie avec succès !


In [15]:

# Définir les données partagées entre les nœuds du Job Search Agent.
# Stocker les résultats de chaque API et les éventuelles erreurs.

class JobSearchState(TypedDict):

    # Critères de recherche
    target_roles: list[str]
    location: Optional[str]
    filters: JobSearchFilters
    search_queries: list[str]

    # Résultats des différentes sources
    jsearch_jobs: list[JobOffer]
    remotive_jobs: list[dict]
    arbeitnow_jobs: list[dict]
    greenhouse_jobs: list[JobOffer]

    # Résultats après traitement
    normalized_jobs: list[JobOffer]
    final_jobs: list[JobOffer]
    total_jobs: int

    # Erreurs éventuelles
    errors: list[str]


# État initial pour tester le graphe
initial_state: JobSearchState = {

    "target_roles": search_filters.keywords,
    "location": search_filters.location,
    "filters": search_filters,
    "search_queries": [],

    "jsearch_jobs": [],
    "remotive_jobs": [],
    "arbeitnow_jobs": [],
    "greenhouse_jobs": [],

    "normalized_jobs": [],
    "final_jobs": [],
    "total_jobs": 0,

    "errors": []
}


print("✅ JobSearchState créé avec succès !")
print("Métiers ciblés :", initial_state["target_roles"])
print("Pays ciblé :", initial_state["filters"].country)
print("Nombre initial d'offres :", initial_state["total_jobs"])


✅ JobSearchState créé avec succès !
Métiers ciblés : ['Python Developer', 'AI Engineer']
Pays ciblé : tn
Nombre initial d'offres : 0


In [16]:

# Préparer les requêtes de recherche à partir des métiers du candidat.
# Enregistrer ces requêtes dans le State partagé de LangGraph.

def prepare_search_query(
    state: JobSearchState
) -> dict:

    roles = state["target_roles"]
    filters = state["filters"]

    location = (
        state["location"]
        or ("Tunisia" if filters.country.lower() == "tn"
            else filters.country)
    )

    search_queries = [
        f"{role} jobs in {location}"
        for role in roles
        if role.strip()
    ]

    print("=== PREPARE SEARCH QUERY ===")

    for query in search_queries:
        print("🔎", query)

    return {
        "search_queries": search_queries
    }


# Test local du nœud, sans appel API
query_result = prepare_search_query(initial_state)

print("\nRésultat retourné :")
print(query_result)


=== PREPARE SEARCH QUERY ===
🔎 Python Developer jobs in Tunisia
🔎 AI Engineer jobs in Tunisia

Résultat retourné :
{'search_queries': ['Python Developer jobs in Tunisia', 'AI Engineer jobs in Tunisia']}


In [17]:

# Exécuter la recherche JSearch avec les filtres du candidat.
# Enregistrer les offres trouvées ou les erreurs dans le State LangGraph.

def search_jsearch(
    state: JobSearchState
) -> dict:

    print("\n=== SEARCH JSEARCH ===")

    filters = state["filters"]

    try:
        offers = fetch_jsearch_jobs(filters)

        print(f"✅ JSearch : {len(offers)} offres trouvées")

        return {
            "jsearch_jobs": offers
        }

    except (requests.RequestException, ValueError, TypeError) as error:

        error_message = f"JSearch : {error}"

        print(f"❌ {error_message}")

        return {
            "jsearch_jobs": [],
            "errors": state["errors"] + [error_message]
        }


print("✅ Nœud search_jsearch défini avec succès !")


✅ Nœud search_jsearch défini avec succès !


In [18]:

# Exécuter la recherche d'offres sur Remotive selon les filtres du candidat.
# Enregistrer les résultats ou les erreurs dans le State LangGraph.

def search_remotive(
    state: JobSearchState
) -> dict:

    print("\n=== SEARCH REMOTIVE ===")

    filters = state["filters"]

    try:
        jobs = fetch_remotive_jobs(filters)

        print(f"✅ Remotive : {len(jobs)} offres trouvées")

        return {
            "remotive_jobs": jobs
        }

    except (requests.RequestException, ValueError, TypeError) as error:

        error_message = f"Remotive : {error}"

        print(f"❌ {error_message}")

        return {
            "remotive_jobs": [],
            "errors": state["errors"] + [error_message]
        }


print("✅ Nœud search_remotive défini avec succès !")


✅ Nœud search_remotive défini avec succès !


In [32]:

# Exécuter la recherche d'offres depuis l'API Arbeitnow.
# Enregistrer les résultats ou les erreurs dans le State LangGraph.

def search_arbeitnow(
    state: JobSearchState
) -> dict:

    print("\n=== SEARCH ARBEITNOW ===")

    filters = state["filters"]

    try:
        jobs = fetch_arbeitnow_jobs(filters)

        print(f"✅ Arbeitnow : {len(jobs)} offres trouvées")

        return {
            "arbeitnow_jobs": jobs
        }

    except (requests.RequestException, ValueError, TypeError) as error:

        error_message = f"Arbeitnow : {error}"

        print(f"❌ {error_message}")

        return {
            "arbeitnow_jobs": [],
            "errors": state["errors"] + [error_message]
        }


print("✅ Nœud search_arbeitnow défini avec succès !")


✅ Nœud search_arbeitnow défini avec succès !


In [20]:

# Rechercher les offres publiées par les entreprises utilisant Greenhouse.
# Enregistrer les offres normalisées ou les erreurs dans le State LangGraph.

def search_greenhouse(
    state: JobSearchState
) -> dict:

    print("\n=== SEARCH GREENHOUSE ===")

    filters = state["filters"]

    try:
        offers = fetch_greenhouse_jobs(filters)

        print(f"✅ Greenhouse : {len(offers)} offres trouvées")

        return {
            "greenhouse_jobs": offers
        }

    except (requests.RequestException, ValueError, TypeError) as error:

        error_message = f"Greenhouse : {error}"

        print(f"❌ {error_message}")

        return {
            "greenhouse_jobs": [],
            "errors": state["errors"] + [error_message]
        }


print("✅ Nœud search_greenhouse défini avec succès !")


✅ Nœud search_greenhouse défini avec succès !


In [21]:

# Regrouper et normaliser les offres récupérées depuis les quatre API.
# Supprimer les doublons pour préparer une liste unique de JobOffer.

def normalize_jobs(
    state: JobSearchState
) -> dict:

    print("\n=== NORMALIZE JOBS ===")

    # 1. Offres déjà normalisées
    jsearch_offers = state["jsearch_jobs"]
    greenhouse_offers = state["greenhouse_jobs"]

    # 2. Normaliser les offres brutes
    remotive_offers = normalize_remotive_jobs(
        state["remotive_jobs"]
    )

    arbeitnow_offers = normalize_arbeitnow_jobs(
        state["arbeitnow_jobs"]
    )

    # 3. Fusionner toutes les sources
    all_offers = (
        jsearch_offers
        + remotive_offers
        + arbeitnow_offers
        + greenhouse_offers
    )

    # 4. Supprimer les doublons
    unique_offers = []
    seen_ids = set()

    for offer in all_offers:

        unique_key = (
            offer.source,
            offer.source_id
        )

        if unique_key in seen_ids:
            continue

        seen_ids.add(unique_key)
        unique_offers.append(offer)

    print("JSearch :", len(jsearch_offers))
    print("Remotive :", len(remotive_offers))
    print("Arbeitnow :", len(arbeitnow_offers))
    print("Greenhouse :", len(greenhouse_offers))
    print("Total avant dédoublonnage :", len(all_offers))
    print("Total après dédoublonnage :", len(unique_offers))

    return {
        "normalized_jobs": unique_offers
    }


print("✅ Nœud normalize_jobs défini avec succès !")


✅ Nœud normalize_jobs défini avec succès !


In [22]:

# Nettoyer les descriptions des offres normalisées pour le Matching Agent.
# Enregistrer les offres finales et leur nombre total dans le State LangGraph.

def prepare_final_jobs(
    state: JobSearchState
) -> dict:

    print("\n=== PREPARE FINAL JOBS ===")

    # 1. Récupérer les offres normalisées
    normalized_offers = state["normalized_jobs"]

    # 2. Nettoyer les descriptions HTML
    final_offers = prepare_jobs_for_matching(
        normalized_offers
    )

    # 3. Calculer le nombre total d'offres
    total = len(final_offers)

    print(f"✅ {total} offres prêtes pour le Matching Agent")

    return {
        "final_jobs": final_offers,
        "total_jobs": total
    }


# Test local sans appel API
test_state = initial_state.copy()

test_state["normalized_jobs"] = []

test_result = prepare_final_jobs(test_state)

print("\nRésultat du test :")
print("Total :", test_result["total_jobs"])
print("Offres finales :", test_result["final_jobs"])



=== PREPARE FINAL JOBS ===
✅ 0 offres prêtes pour le Matching Agent

Résultat du test :
Total : 0
Offres finales : []


In [23]:

# Construire le graphe LangGraph en reliant les nœuds de recherche.
# Définir l'ordre d'exécution depuis START jusqu'à END.

job_search_builder = StateGraph(JobSearchState)

# 1. Ajouter les nœuds
job_search_builder.add_node(
    "prepare_search_query",
    prepare_search_query
)

job_search_builder.add_node(
    "search_jsearch",
    search_jsearch
)

job_search_builder.add_node(
    "search_remotive",
    search_remotive
)

job_search_builder.add_node(
    "search_arbeitnow",
    search_arbeitnow
)

job_search_builder.add_node(
    "search_greenhouse",
    search_greenhouse
)

job_search_builder.add_node(
    "normalize_jobs",
    normalize_jobs
)

job_search_builder.add_node(
    "prepare_final_jobs",
    prepare_final_jobs
)

# 2. Relier les nœuds
job_search_builder.add_edge(
    START,
    "prepare_search_query"
)

job_search_builder.add_edge(
    "prepare_search_query",
    "search_jsearch"
)

job_search_builder.add_edge(
    "search_jsearch",
    "search_remotive"
)

job_search_builder.add_edge(
    "search_remotive",
    "search_arbeitnow"
)

job_search_builder.add_edge(
    "search_arbeitnow",
    "search_greenhouse"
)

job_search_builder.add_edge(
    "search_greenhouse",
    "normalize_jobs"
)

job_search_builder.add_edge(
    "normalize_jobs",
    "prepare_final_jobs"
)

job_search_builder.add_edge(
    "prepare_final_jobs",
    END
)

# 3. Compiler le graphe
job_search_graph = job_search_builder.compile()

print("✅ Graphe Job Search Agent compilé avec succès !")
print("Nombre de nœuds : 7")


✅ Graphe Job Search Agent compilé avec succès !
Nombre de nœuds : 7


In [33]:

# Exécuter le Job Search Agent avec les critères définis précédemment.
# Afficher les offres finales, leur provenance et les erreurs éventuelles.

print("🚀 DÉMARRAGE DU JOB SEARCH AGENT")

# Créer un état neuf pour cette exécution
run_state = {
    **initial_state,
    "search_queries": [],
    "jsearch_jobs": [],
    "remotive_jobs": [],
    "arbeitnow_jobs": [],
    "greenhouse_jobs": [],
    "normalized_jobs": [],
    "final_jobs": [],
    "total_jobs": 0,
    "errors": []
}

# Exécuter le graphe complet
final_state = job_search_graph.invoke(run_state)

print("\n=== RÉSULTAT FINAL ===")
print("Total des offres :", final_state["total_jobs"])

print("\n=== OFFRES TROUVÉES ===")

for index, offer in enumerate(
    final_state["final_jobs"][:10],
    start=1
):
    print(f"\n--- Offre {index} ---")
    print("Source :", offer.source)
    print("Titre :", offer.title)
    print("Entreprise :", offer.company)
    print("Localisation :", offer.location)
    print("URL :", offer.url)

print("\n=== ERREURS ===")

if final_state["errors"]:
    for error in final_state["errors"]:
        print("❌", error)
else:
    print("✅ Aucune erreur signalée")


🚀 DÉMARRAGE DU JOB SEARCH AGENT
=== PREPARE SEARCH QUERY ===
🔎 Python Developer jobs in Tunisia
🔎 AI Engineer jobs in Tunisia

=== SEARCH JSEARCH ===
✅ JSearch : 0 offres trouvées

=== SEARCH REMOTIVE ===
Recherche Remotive : Python Developer
Recherche Remotive : AI Engineer
✅ Remotive : 14 offres trouvées

=== SEARCH ARBEITNOW ===
✅ Arbeitnow : 7 offres trouvées

=== SEARCH GREENHOUSE ===
✅ Greenhouse : 1 offres trouvées

=== NORMALIZE JOBS ===
JSearch : 0
Remotive : 14
Arbeitnow : 7
Greenhouse : 1
Total avant dédoublonnage : 22
Total après dédoublonnage : 22

=== PREPARE FINAL JOBS ===
✅ 22 offres prêtes pour le Matching Agent

=== RÉSULTAT FINAL ===
Total des offres : 22

=== OFFRES TROUVÉES ===

--- Offre 1 ---
Source : remotive
Titre : Remote Office Assistant
Entreprise : Coalition Technologies 
Localisation : Worldwide
URL : https://remotive.com/remote-jobs/marketing/remote-office-assistant-1680495

--- Offre 2 ---
Source : remotive
Titre : Inside Sales Contractor
Entreprise : Cr

In [27]:

print("fetch_arbeitnow_jobs :", "fetch_arbeitnow_jobs" in globals())
print("fetch_remotive_jobs :", "fetch_remotive_jobs" in globals())
print("fetch_jsearch_jobs :", "fetch_jsearch_jobs" in globals())
print("fetch_greenhouse_jobs :", "fetch_greenhouse_jobs" in globals())


fetch_arbeitnow_jobs : False
fetch_remotive_jobs : True
fetch_jsearch_jobs : True
fetch_greenhouse_jobs : True


In [34]:

# Vérifier les résultats JSearch déjà enregistrés dans le State.
# Confirmer les autres résultats sans effectuer de nouvel appel API.

print("=== DIAGNOSTIC JOB SEARCH AGENT ===")

print("JSearch :", len(final_state["jsearch_jobs"]))
print("Remotive :", len(final_state["remotive_jobs"]))
print("Arbeitnow :", len(final_state["arbeitnow_jobs"]))
print("Greenhouse :", len(final_state["greenhouse_jobs"]))

print("\nTotal final :", final_state["total_jobs"])
print("Erreurs :", final_state["errors"])

print("\n=== SOURCES DES OFFRES FINALES ===")

from collections import Counter

source_counts = Counter(
    offer.source
    for offer in final_state["final_jobs"]
)

for source, count in source_counts.items():
    print(f"{source} : {count}")


=== DIAGNOSTIC JOB SEARCH AGENT ===
JSearch : 0
Remotive : 14
Arbeitnow : 7
Greenhouse : 1

Total final : 22
Erreurs : []

=== SOURCES DES OFFRES FINALES ===
remotive : 14
arbeitnow : 7
greenhouse : 1


In [36]:

# Vérifier la lecture d'une réponse JSearch sans utiliser RapidAPI.
# Simuler les deux structures JSON possibles.

test_responses = [
    {"data": [{"job_id": "123"}]},
    {"data": {"jobs": [{"job_id": "456"}]}}
]

for response_data in test_responses:
    payload = response_data.get("data", [])

    if isinstance(payload, list):
        jobs = payload
    elif isinstance(payload, dict):
        jobs = payload.get("jobs", [])
    else:
        jobs = []

    print("Offres détectées :", len(jobs))


Offres détectées : 1
Offres détectées : 1


In [37]:

# Tester uniquement JSearch sans relancer les autres API.
# Afficher les offres reçues et les erreurs éventuelles.

print("=== TEST JSEARCH ISOLÉ ===")

try:
    jsearch_test_offers = fetch_jsearch_jobs(search_filters)

    print("Offres normalisées :", len(jsearch_test_offers))

    for offer in jsearch_test_offers[:5]:
        print("\nTitre :", offer.title)
        print("Entreprise :", offer.company)
        print("Localisation :", offer.location)

except Exception as error:
    print("❌ Erreur JSearch :", type(error).__name__, error)


=== TEST JSEARCH ISOLÉ ===
JSearch : 0 offres brutes reçues
Offres normalisées : 0


In [38]:

# Examiner la structure réelle de la réponse JSearch V2.
# Afficher les informations de diagnostic sans exposer la clé RapidAPI.

print("=== DIAGNOSTIC RÉPONSE JSEARCH ===")

params = {
    "query": "Python Developer in Tunisia",
    "country": "tn",
    "num_pages": 1
}

headers = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": RAPIDAPI_HOST
}

try:
    response = requests.get(
        JSEARCH_API_URL,
        headers=headers,
        params=params,
        timeout=40
    )

    print("Statut HTTP :", response.status_code)

    data = response.json()

    print("Type de réponse :", type(data).__name__)

    if isinstance(data, dict):
        print("Clés principales :", list(data.keys()))

        for key, value in data.items():
            print(
                f"{key} : type={type(value).__name__}",
                f"| taille={len(value) if isinstance(value, (list, dict)) else 'N/A'}"
            )

        print("Message API :", data.get("message"))
        print("Status API :", data.get("status"))

    elif isinstance(data, list):
        print("Nombre d'éléments :", len(data))

except Exception as error:
    print("Erreur :", type(error).__name__, error)


=== DIAGNOSTIC RÉPONSE JSEARCH ===
Statut HTTP : 200
Type de réponse : dict
Clés principales : ['status', 'request_id', 'parameters', 'data']
status : type=str | taille=N/A
request_id : type=str | taille=N/A
parameters : type=dict | taille=4
data : type=dict | taille=2
Message API : None
Status API : OK


In [39]:

# Examiner les champs internes de la réponse JSearch déjà récupérée.
# Identifier où se trouvent les offres sans consommer de quota API.

print("=== STRUCTURE INTERNE JSEARCH ===")

payload = data.get("data", {})

print("Clés de data :", list(payload.keys()))

for key, value in payload.items():

    print(f"\nChamp : {key}")
    print("Type :", type(value).__name__)

    if isinstance(value, list):
        print("Nombre d'éléments :", len(value))

        if value and isinstance(value[0], dict):
            print("Clés du premier élément :", list(value[0].keys()))

    elif isinstance(value, dict):
        print("Sous-clés :", list(value.keys()))

        for subkey, subvalue in value.items():
            if isinstance(subvalue, list):
                print(f"  {subkey} : {len(subvalue)} éléments")

    else:
        print("Valeur :", str(value)[:200])


=== STRUCTURE INTERNE JSEARCH ===
Clés de data : ['jobs', 'cursor']

Champ : jobs
Type : list
Nombre d'éléments : 2
Clés du premier élément : ['job_id', 'job_title', 'employer_name', 'employer_logo', 'employer_website', 'job_publisher', 'job_employment_type', 'job_employment_types', 'job_apply_link', 'job_apply_is_direct', 'apply_options', 'job_description', 'job_is_remote', 'job_posted_at', 'job_posted_at_timestamp', 'job_posted_at_datetime_utc', 'job_location', 'job_city', 'job_state', 'job_country', 'job_latitude', 'job_longitude', 'job_benefits', 'job_benefits_strings', 'job_google_link', 'job_salary', 'job_salary_string', 'job_min_salary', 'job_max_salary', 'job_salary_period', 'job_highlights', 'job_onet_soc', 'job_onet_job_zone', 'employer_reviews', 'job_uid']

Champ : cursor
Type : NoneType
Valeur : None


In [40]:

# Normaliser les offres JSearch déjà récupérées dans la réponse API.
# Vérifier que les offres réelles deviennent des objets JobOffer valides.

print("=== TEST NORMALISATION JSEARCH ===")

raw_jsearch_jobs = data["data"]["jobs"]

normalized_jsearch_offers = normalize_jsearch_jobs(
    raw_jsearch_jobs
)

print("Offres brutes :", len(raw_jsearch_jobs))
print("Offres normalisées :", len(normalized_jsearch_offers))

for offer in normalized_jsearch_offers:
    print("\n---------------------------")
    print("Titre :", offer.title)
    print("Entreprise :", offer.company)
    print("Localisation :", offer.location)
    print("URL :", offer.url)


=== TEST NORMALISATION JSEARCH ===
Offres brutes : 2
Offres normalisées : 2

---------------------------
Titre : Ingénieur Backend Python
Entreprise : Iguane Solutions
Localisation : تونس     •  عبر Trabajo.org
URL : https://tn.trabajo.org/offre-4103-d75263a6797870af325db1a8d38e2545

---------------------------
Titre : Senior Python / Django Developer
Entreprise : Dune Technology
Localisation : تونس     •  عبر BeBee
URL : https://bebee.com/tn/jobs/senior-python-django-developer-dune-technology-tunis--ss-tn-9ycn7c


In [43]:

# Ajouter les offres JSearch déjà récupérées aux résultats du graphe.
# Supprimer les doublons et préparer les offres pour le Matching Agent.

print("=== FUSION DES OFFRES ===")

all_offers = (
    final_state["final_jobs"]
    + normalized_jsearch_offers
)

unique_offers = []
seen_ids = set()

for offer in all_offers:

    unique_key = (offer.source, offer.source_id)

    if unique_key in seen_ids:
        continue

    seen_ids.add(unique_key)
    unique_offers.append(offer)

# Nettoyer les descriptions et localisations
for offer in unique_offers:
    if offer.source == "jsearch":
        offer.location = clean_jsearch_location(
            offer.location
        )

final_offers = prepare_jobs_for_matching(unique_offers)

# Mettre à jour le résultat final
final_state["jsearch_jobs"] = normalized_jsearch_offers
final_state["normalized_jobs"] = unique_offers
final_state["final_jobs"] = final_offers
final_state["total_jobs"] = len(final_offers)

print("Total des offres :", final_state["total_jobs"])

for source, count in Counter(
    offer.source for offer in final_offers
).items():
    print(f"{source} : {count}")

print("\n✅ Offres prêtes pour le Matching Agent !")


=== FUSION DES OFFRES ===
Total des offres : 24
remotive : 14
arbeitnow : 7
greenhouse : 1
jsearch : 2

✅ Offres prêtes pour le Matching Agent !


In [44]:

# Sauvegarder les offres finales dans un fichier JSON.
# Permettre au Matching Agent de les réutiliser sans rappeler les API.

import json

output_path = Path.cwd().parent / "data" / "jobs_for_matching.json"

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

jobs_data = [
    offer.model_dump(mode="json")
    for offer in final_state["final_jobs"]
]

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(
        jobs_data,
        file,
        ensure_ascii=False,
        indent=2
    )

print("✅ Sauvegarde réussie !")
print("Fichier :", output_path)
print("Nombre d'offres :", len(jobs_data))


✅ Sauvegarde réussie !
Fichier : c:\Users\moham\Desktop\PFA IGL4\S1\Python\data\jobs_for_matching.json
Nombre d'offres : 24
